How patient dataset is created?

Since publicly available datasets with patient profiles and personalized nutrition targets are limited, I generated a synthetic dataset using medically plausible ranges and rule-based constraints. BMI was computed from height and weight to maintain consistency, and nutritional targets were derived from dietary guidelines based on health condition and activity level

In [2]:
import pandas as pd
import numpy as np

Instead of hardcoding values throughout the notebook, define them once.

You only change one variable.

This is called parameterization, and it's a good coding practice.

In [ ]:
np.random.seed(42)

#you (and anyone reviewing your project) get the same synthetic dataset every time, which is essential for reproducible experiments.

Children require completely different nutritional guidelines. This will focus on adults.

In [3]:
NUM_PATIENTS = 20000
ages = np.random.randint(
    low=18,
    high=81,
    size=NUM_PATIENTS
)

Generate gender

In [4]:
genders = np.random.choice(
    ["Male", "Female"],
    size=NUM_PATIENTS,
    p=[0.5, 0.5]
)

In [6]:
print(ages[:10])
print(ages.min())
print(ages.max())

[71 56 34 79 67 35 28 19 32 55]
18
80


generate heights

Male

Typical range:

1.60 m  →  1.90 m

Female

Typical range:

1.50 m  →  1.80 m

In [7]:
heights = []

for gender in genders:

    if gender == "Male":
        heights.append(
            round(np.random.uniform(1.60, 1.90), 2)
        )

    else:
        heights.append(
            round(np.random.uniform(1.50, 1.80), 2)
        )

heights = np.array(heights)

In [8]:
print(heights[:10])

print(heights.min())

print(heights.max())

[1.61 1.79 1.72 1.68 1.65 1.55 1.52 1.76 1.78 1.71]
1.5
1.9


Height = 1.52 m

Weight = 118 kg

or

Height = 1.90 m

Weight = 46 kg

These combinations may be valid in rare cases, but if generated randomly they'll produce an unrealistic distribution.

Instead of generating weight,

generate a realistic target BMI first.

Weight = BMI × Height²

This guarantees:

Realistic heights

Realistic weights

Perfect BMI consistency

| Category    | BMI Range   | Probability |
| ----------- | ----------- | ----------: |
| Underweight | 17.0 - 18.4 |          5% |
| Normal      | 18.5 - 24.9 |         45% |
| Overweight  | 25.0 - 29.9 |         30% |
| Obese       | 30.0 - 39.0 |         20% |


In [9]:
bmi_categories = np.random.choice(
    ["Underweight", "Normal", "Overweight", "Obese"],
    size=NUM_PATIENTS,
    p=[0.05, 0.45, 0.30, 0.20]
)

In [10]:
bmi = []

for category in bmi_categories:

    if category == "Underweight":
        bmi.append(round(np.random.uniform(17.0, 18.4), 1))

    elif category == "Normal":
        bmi.append(round(np.random.uniform(18.5, 24.9), 1))

    elif category == "Overweight":
        bmi.append(round(np.random.uniform(25.0, 29.9), 1))

    else:
        bmi.append(round(np.random.uniform(30.0, 39.0), 1))

bmi = np.array(bmi)

In [11]:
print(bmi[:10])

print(bmi.min())

print(bmi.max())

[26.5 19.1 19.3 25.9 19.8 26.2 32.3 21.4 27.5 20.5]
17.0
39.0


In [12]:
pd.Series(bmi_categories).value_counts(normalize=True) * 100

Normal         44.910
Overweight     30.355
Obese          19.755
Underweight     4.980
Name: proportion, dtype: float64

Let's compute the weights

In [13]:
weights = np.round(bmi * (heights ** 2), 1)

In [14]:
print(weights[:10])

print(weights.min())

print(weights.max())

[68.7 61.2 57.1 73.1 53.9 62.9 74.6 66.3 87.1 59.9]
38.8
139.7


In [15]:
calculated_bmi = np.round(weights / (heights ** 2), 1)

In [16]:
print(bmi[:10])

print(calculated_bmi[:10])

[26.5 19.1 19.3 25.9 19.8 26.2 32.3 21.4 27.5 20.5]
[26.5 19.1 19.3 25.9 19.8 26.2 32.3 21.4 27.5 20.5]


In [17]:
print(pd.Series(weights).describe())

count    20000.000000
mean        74.682285
std         18.220718
min         38.800000
25%         60.900000
50%         72.000000
75%         86.000000
max        139.700000
dtype: float64


In [19]:
weights.describe()

AttributeError: 'numpy.ndarray' object has no attribute 'describe'

In [20]:
pd.Series(weights).describe()

count    20000.000000
mean        81.687850
std         21.590114
min         45.000000
25%         63.000000
50%         81.000000
75%        100.000000
max        119.000000
dtype: float64

BMI influence the disease probabilities.

BMI

 │

 ├── Underweight

 │      └── Mostly Healthy

 │

 ├── Normal

 │      └── Mostly Healthy

 │

 ├── Overweight

 │      └── Diabetes / Hypertension become more common
 
 │
 
 └── Obese
 
        └── Obesity becomes dominant

In [21]:
health_conditions = []

In [22]:
for value in bmi:

    if value < 18.5:
        condition = np.random.choice(
            ["Healthy", "Underweight"],
            p=[0.8, 0.2]
        )

    elif value < 25:
        condition = np.random.choice(
            ["Healthy", "Diabetes", "Hypertension", "Heart Disease"],
            p=[0.85, 0.05, 0.05, 0.05]
        )

    elif value < 30:
        condition = np.random.choice(
            ["Healthy", "Diabetes", "Hypertension", "Obesity"],
            p=[0.35, 0.25, 0.20, 0.20]
        )

    else:
        condition = np.random.choice(
            ["Obesity", "Diabetes", "Hypertension", "Heart Disease"],
            p=[0.55, 0.20, 0.15, 0.10]
        )

    health_conditions.append(condition)

In [23]:
health_conditions = np.array(health_conditions)

In [24]:
pd.Series(health_conditions).value_counts()

Healthy          10589
Obesity           3373
Diabetes          2775
Hypertension      2243
Heart Disease      828
Underweight        192
Name: count, dtype: int64

How diseases are assigned?

I used a rule-based probabilistic generator. Instead of assigning diseases randomly, BMI influenced the probability distribution of health conditions. This preserved realistic relationships in the synthetic dataset while still maintaining variability

activity level depend slightly on age.

Generally:

Younger people tend to be more active.

Middle-aged people tend to have moderate activity.

Older adults are more likely to have low activity.

This doesn't mean every young person is active—it just changes the probabilities.

In [26]:
activity_levels = []

In [27]:
for age in ages:

    if age < 30:
        activity = np.random.choice(
            ["Low", "Medium", "High"],
            p=[0.20, 0.40, 0.40]
        )

    elif age < 50:
        activity = np.random.choice(
            ["Low", "Medium", "High"],
            p=[0.30, 0.50, 0.20]
        )

    else:
        activity = np.random.choice(
            ["Low", "Medium", "High"],
            p=[0.50, 0.35, 0.15]
        )

    activity_levels.append(activity)

In [28]:
activity_levels = np.array(activity_levels)

In [29]:
pd.Series(activity_levels).value_counts()

Medium    8223
Low       7523
High      4254
Name: count, dtype: int64

| Diet Type      | Probability |
| -------------- | ----------: |
| Vegetarian     |         40% |
| Non-Vegetarian |         50% |
| Vegan          |         10% |

The diet type is a personal preference or lifestyle, not the disease itself.

So we'll generate diet type independently with realistic proportions.

In [30]:
diet_types = np.random.choice(
    ["Vegetarian", "Non-Vegetarian", "Vegan"],
    size=NUM_PATIENTS,
    p=[0.40, 0.50, 0.10]
)

In [31]:
pd.Series(diet_types).value_counts()

Non-Vegetarian    9962
Vegetarian        8032
Vegan             2006
Name: count, dtype: int64

Most people don't have any food allergy.

| Allergy | Probability |
| ------- | ----------: |
| None    |         75% |
| Dairy   |          8% |
| Gluten  |          6% |
| Peanut  |          5% |
| Seafood |          4% |
| Soy     |          2% |


In [32]:
allergies = np.random.choice(
    ["None", "Dairy", "Gluten", "Peanut", "Seafood", "Soy"],
    size=NUM_PATIENTS,
    p=[0.75, 0.08, 0.06, 0.05, 0.04, 0.02]
)

In [33]:
pd.Series(allergies).value_counts()

None       14962
Dairy       1617
Gluten      1182
Peanut      1110
Seafood      750
Soy          379
Name: count, dtype: int64

combine everything

In [34]:
patient_df = pd.DataFrame({
    "age": ages,
    "gender": genders,
    "height_m": heights,
    "weight_kg": weights,
    "bmi": bmi,
    "health_condition": health_conditions,
    "activity_level": activity_levels,
    "diet_type": diet_types,
    "allergy": allergies
})

In [35]:
patient_df.head()

,age,gender,height_m,weight_kg,bmi,health_condition,activity_level,diet_type,allergy
0,71,Female,1.61,112,26.5,Diabetes,Low,Non-Vegetarian,None
1,56,Female,1.79,111,19.1,Healthy,Medium,Non-Vegetarian,None
2,34,Female,1.72,96,19.3,Healthy,Low,Vegan,Dairy
3,79,Female,1.68,114,25.9,Healthy,Medium,Vegetarian,Dairy
4,67,Male,1.65,47,19.8,Hypertension,Low,Non-Vegetarian,Seafood


In [36]:
patient_df.isnull().sum()

age                 0
gender              0
height_m            0
weight_kg           0
bmi                 0
health_condition    0
activity_level      0
diet_type           0
allergy             0
dtype: int64

Save the patient profile dataset

In [37]:
patient_df.to_csv(
    "../data/processed/patient_profiles.csv",
    index=False
)